# CSV File Handling in Python

CSV means comma-separated values. A CSV file stores table-like data as plain text, which makes it easy to share between Python, spreadsheets, databases, and many other tools.

This notebook covers:

- writing CSV files with Python's built-in `csv` module
- reading CSV rows as lists
- reading and writing rows as dictionaries
- handling delimiters, missing values, and basic validation
- using `pandas` as an optional tool for larger data analysis workflows

## 1. Setup

The built-in `csv` module is enough for many file tasks and does not require installation.

`pandas` is useful for analysis, cleaning, grouping, and filtering. In this notebook, pandas examples are optional and will be skipped if pandas is not installed.

In [ ]:
import csv
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

example_dir = Path("csv_examples")
example_dir.mkdir(exist_ok=True)

students_path = example_dir / "students.csv"
products_path = example_dir / "products.csv"
dirty_path = example_dir / "students_with_missing_values.csv"
semicolon_path = example_dir / "semicolon_scores.csv"
sales_path = example_dir / "sales.csv"

print(f"Example folder: {example_dir.resolve()}")
print(f"Pandas available: {pd is not None}")

Example folder: C:\git\Python_Training\07 - Working with Files\csv_examples
Pandas available: False


In [8]:
if pd is None:
    print("Optional pandas examples will be skipped.")
    print("To enable them later, install pandas in the notebook environment.")
else:
    print("Pandas is ready for the optional examples.")

Optional pandas examples will be skipped.
To enable them later, install pandas in the notebook environment.


## 2. Basic CSV handling with the `csv` module

Use the built-in `csv` module when you need lightweight, dependency-free CSV reading and writing.

### 2.1 Writing rows as lists

`csv.writer` writes each row from a list or tuple.

Use `newline=""` when opening CSV files for writing. This prevents extra blank lines on Windows.

In [9]:
students = [
    ["student_id", "name", "grade"],
    [1, "Alice", "A"],
    [2, "Bob", "B"],
    [3, "Charlie", "A-"],
]

with open(students_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(students)

print(f"Wrote {students_path}")

Wrote csv_examples\students.csv


### 2.2 Reading rows as lists

`csv.reader` reads each CSV row as a list of strings.

Even values that look numeric are read as strings unless you convert them yourself.

In [10]:
with open(students_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

['student_id', 'name', 'grade']
['1', 'Alice', 'A']
['2', 'Bob', 'B']
['3', 'Charlie', 'A-']


### 2.3 Reading and writing dictionaries

`csv.DictWriter` writes dictionaries using named columns.

`csv.DictReader` reads each row as a dictionary, which makes the code easier to understand when the CSV has headers.

In [11]:
product_rows = [
    {"product": "Laptop", "price": 999.99, "stock": 10},
    {"product": "Mouse", "price": 19.99, "stock": 100},
    {"product": "Keyboard", "price": 49.99, "stock": 50},
]

fieldnames = ["product", "price", "stock"]

with open(products_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(product_rows)

with open(products_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        stock = int(row["stock"])
        print(f"{row['product']}: {stock} units")

Laptop: 10 units
Mouse: 100 units
Keyboard: 50 units


## 3. Common CSV issues

Real CSV files often contain missing values, unexpected delimiters, or values that need conversion.

The built-in `csv` module can handle these problems, but you need to be explicit about what should happen.

### 3.1 Missing values and basic cleaning

CSV files do not have a real `None` value. Missing values usually appear as empty strings.

A common cleanup step is to replace empty strings with a default value.

In [12]:
dirty_rows = [
    {"name": "Alice", "age": "25", "city": "Paris"},
    {"name": "Bob", "age": "", "city": "London"},
    {"name": "", "age": "30", "city": "Berlin"},
]

with open(dirty_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "age", "city"])
    writer.writeheader()
    writer.writerows(dirty_rows)

with open(dirty_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    cleaned_rows = []
    for row in reader:
        cleaned_rows.append({
            "name": row["name"] or "Unknown",
            "age": int(row["age"] or 0),
            "city": row["city"],
        })

cleaned_rows

[{'name': 'Alice', 'age': 25, 'city': 'Paris'},
 {'name': 'Bob', 'age': 0, 'city': 'London'},
 {'name': 'Unknown', 'age': 30, 'city': 'Berlin'}]

### 3.2 Different delimiters

Not every CSV uses commas. Some files use semicolons, tabs, or pipes.

Pass `delimiter=";"` or another delimiter to read those files correctly.

In [13]:
semicolon_data = "name;score\nAlice;92\nBob;85\nCharlie;88\n"
semicolon_path.write_text(semicolon_data, encoding="utf-8")

with open(semicolon_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file, delimiter=";")
    for row in reader:
        print(f"{row['name']} scored {row['score']}")

Alice scored 92
Bob scored 85
Charlie scored 88


### 3.3 Converting and summarizing values

CSV readers return strings. Convert numeric columns before doing math.

This example writes small sales data, reads it back, converts revenue to `float`, and summarizes revenue by product.

In [14]:
sales_rows = [
    {"date": "2026-01-01", "product": "Laptop", "revenue": "999.99"},
    {"date": "2026-01-02", "product": "Mouse", "revenue": "19.99"},
    {"date": "2026-01-03", "product": "Laptop", "revenue": "899.99"},
]

with open(sales_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["date", "product", "revenue"])
    writer.writeheader()
    writer.writerows(sales_rows)

revenue_by_product = {}

with open(sales_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        product = row["product"]
        revenue = float(row["revenue"])
        revenue_by_product[product] = revenue_by_product.get(product, 0) + revenue

for product, revenue in revenue_by_product.items():
    print(f"{product}: ${revenue:.2f}")

Laptop: $1899.98
Mouse: $19.99


## 4. Optional pandas workflow

`pandas` is excellent for filtering, grouping, joining, and larger analysis tasks.

The next cell runs only if pandas is installed. If pandas is not installed, the notebook continues without error.

In [15]:
if pd is None:
    print("Skipping pandas example because pandas is not installed.")
else:
    sales_df = pd.read_csv(sales_path)
    sales_df["revenue"] = sales_df["revenue"].astype(float)

    print("Total revenue:", sales_df["revenue"].sum())
    print("\nRevenue by product:")
    print(sales_df.groupby("product")["revenue"].sum())

Skipping pandas example because pandas is not installed.


## 5. Best practices and common choices

### Key recommendations

1. Always specify `newline=""` when opening CSV files in write mode.
2. Use `encoding="utf-8"` for text files unless you have a specific reason not to.
3. Use `DictReader` and `DictWriter` when the CSV has headers.
4. Convert values after reading because CSV data starts as text.
5. Validate required columns before processing a file.
6. Use pandas when you need richer analysis, filtering, grouping, joins, or large data tools.
7. Do not store passwords, API keys, or sensitive personal data in plain-text CSV files.

| Scenario | Good choice |
|---|---|
| Simple read/write | `csv` module |
| Dependency-free scripts | `csv` module |
| Header-based rows | `csv.DictReader` / `csv.DictWriter` |
| Cleaning, grouping, joining | `pandas` |
| Very large files | chunked processing |

## 6. Practice

### Basic

1. Create a CSV file named `employees.csv` in `csv_examples` with columns `name`, `department`, and `salary`.
2. Read the file with `csv.DictReader` and print only employees in the `Engineering` department.
3. Convert `salary` to `float` and calculate the average salary.

### Intermediate

4. Create a semicolon-separated file and read it with the correct delimiter.
5. Add a validation step that checks whether required columns are present before processing a CSV file.

### Challenge

6. Create two files: `students.csv` and `courses.csv`. Join them by `student_id` using dictionaries, then print each student's name with their course.

## 7. Summary

You now know how to:

- read and write CSV files with Python's built-in `csv` module
- use `newline=""` and `encoding="utf-8"` for reliable text file handling
- read rows as lists with `csv.reader`
- read and write named rows with `DictReader` and `DictWriter`
- handle missing values and custom delimiters
- convert text values before doing numeric calculations
- use pandas as an optional tool for analysis workflows